In [1]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_example
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,44,False)#OUTPUT_DIM_NOTES,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
#cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch03_valAcc0.9456_valPrec0.7303_valRecall0.3329.weights.h5')#
cnn_model.load_weights('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training_subset/filtered_poly_data0.tfrecord'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_example).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

I0000 00:00:1773482823.041313   16241 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773482823.479656   16241 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773482824.533675   16241 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1773482826.258027   16241 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel bi

Initial input shape: (None, 148, 512)
After first Conv2D: (None, 74, 64)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 148, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 148, 256,  │          0 │ input_layer[0][0] │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 148, 32,   │        272 │ reshape[0][0]     │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 148, 32,   │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 148, 32,   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d   │ (None, 148, 32,   │          0 │ leaky_re_lu[0][0] │
│ (SpatialDropout2D)  │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 148, 512)  │          0 │ spatial_dropout2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 148, 512)  │    263,040 │ reshape_1[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 148, 512)  │          0 │ reshape_1[0][0],  │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 148, 512)  │      1,024 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 148, 256)  │    131,328 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 148, 512)  │    131,584 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 148, 512)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 148, 512)  │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 148, 512)  │      1,024 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 148, 32)   │    114,720 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 148, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 148, 32)   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,388,580 (5.30 MB)

...

INFO:tensorflow:Assets written to: /tmp/tmpvswwj09c/assets


Saved artifact at '/tmp/tmpvswwj09c'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 148, 256, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 44), dtype=tf.float32, name=None)
Captures:
  132122393546576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122393547728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122393545424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122393545808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122393547152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122393547536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122393548304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122393549264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122393549072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132122393549648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1321223935494

/home/gerald/anaconda3/envs/gpu/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1773482832.031516   16241 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773482832.031531   16241 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1773482832.032152   16241 reader.cc:83] Reading SavedModel from: /tmp/tmpvswwj09c
I0000 00:00:1773482832.033523   16241 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1773482832.033529   16241 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpvswwj09c
I0000 00:00:1773482832.048803   16241 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1773482832.052063   16241 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1773482832.161443   16241 loader.cc:220] Running initialization op on S

TFLite model saved as guitarmidi.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1773482834.143702   16241 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.
